In [13]:
import re, os, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.manifold import TSNE
warnings.filterwarnings("ignore")
os.makedirs("/tmp/mod05", exist_ok=True)
plt.rcParams.update({"figure.dpi":120,"figure.facecolor":"white",
                     "axes.spines.top":False,"axes.spines.right":False})
np.random.seed(42)

try:
    from gensim.models import Word2Vec
    GENSIM_OK = True
    print("gensim available")
except ImportError:
    GENSIM_OK = False
    print("pip install gensim")

gensim available


In [8]:
%pip install gensim

   ---------------------------------------- 0.0/24.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/24.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/24.4 MB ? eta -:--:--
   ---------- ----------------------------- 6.6/24.4 MB 39.8 MB/s eta 0:00:01
   ------------------------- -------------- 15.7/24.4 MB 41.8 MB/s eta 0:00:01
   ---------------------------------------  24.1/24.4 MB 41.9 MB/s eta 0:00:01
   ---------------------------------------- 24.4/24.4 MB 33.5 MB/s  0:00:01
Note: you may need to restart the kernel to use updated packages.


In [16]:
CLINICAL_SENTENCES = [
    "heart failure reduced ejection fraction BNP elevated furosemide diuresis daily weights",
    "myocardial infarction troponin elevated EKG ST elevation cath lab PCI stent coronary",
    "atrial fibrillation rate control metoprolol digoxin anticoagulation warfarin stroke prevention",
    "hypertension lisinopril amlodipine blood pressure end organ damage kidney function",
    "cardiac catheterization coronary artery disease stenosis stent drug eluting stent",
    "COPD exacerbation albuterol ipratropium bronchospasm wheezes oxygen supplementation SpO2",
    "pneumonia consolidation fever WBC elevated ceftriaxone azithromycin blood cultures",
    "pulmonary embolism anticoagulation rivaroxaban heparin DVT bilateral lower extremity",
    "respiratory failure mechanical ventilation intubation PEEP FiO2 arterial blood gas",
    "pleural effusion thoracentesis protein LDH transudative exudative Light criteria",
    "diabetic ketoacidosis insulin drip anion gap metabolic acidosis bicarbonate potassium",
    "type 2 diabetes metformin GLP-1 HbA1c glycemic control SGLT2 cardiovascular renal benefit",
    "hypoglycemia glucose dextrose altered mental status glucagon diabetes education",
    "acute kidney injury creatinine elevated BUN nephrotoxins hold contrast ACE inhibitor",
    "hemodialysis end stage renal disease potassium hyperkalemia access fistula bicarbonate",
    "nephrotic syndrome proteinuria edema albumin low diuretics renal biopsy immunosuppression",
    "urinary tract infection pyuria nitrites E coli culture sensitivity trimethoprim nitrofurantoin",
    "septic shock vasopressor norepinephrine broad spectrum antibiotics vancomycin meropenem lactate",
    "bacteremia blood cultures Staph aureus MSSA nafcillin vancomycin echocardiogram endocarditis",
    "neutropenic fever chemotherapy immunosuppressed filgrastim antifungal fluconazole cultures",
    "ischemic stroke MRI DWI diffusion restriction tPA thrombolysis antiplatelet aspirin",
    "seizure antiepileptic levetiracetam valproate EEG epilepsy status epilepticus",
    "pain management opioid morphine acetaminophen ibuprofen NSAID multimodal analgesia",
    "deep vein thrombosis anticoagulation enoxaparin warfarin compression ambulation prophylaxis",
    "post-operative fever wound infection cellulitis erythema warmth drainage cultures antibiotics",
    "malnutrition albumin pre-albumin total parenteral nutrition nasogastric tube dietitian",
]

def tokenize(text):
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s-]',' ', text)
    return text.split()

augmented = []
for sent in CLINICAL_SENTENCES:
    base = tokenize(sent)
    augmented.append(base)
    for _ in range(20):
        shuffled = base.copy()
        np.random.shuffle(shuffled)
        augmented.append(shuffled[:max(4, len(shuffled)//2+ np.random.randint(1,5))])

print(f"Training corpus: {len(augmented)} sentences")
print(f"Sample: {augmented[0][:10]}")

Training corpus: 546 sentences
Sample: ['heart', 'failure', 'reduced', 'ejection', 'fraction', 'bnp', 'elevated', 'furosemide', 'diuresis', 'daily']


## Clinical Word2Vec

In [20]:
Sg_model = Word2Vec(sentences=augmented, vector_size=100, window=5, min_count=2, workers=4, sg=1, negative=10, epochs=40, seed=42)

wv = Sg_model.wv
print(f"Vocabulary: {len(wv.key_to_index)} tokens | Dim: {wv.vector_size}")
print("\nSemantic similarity examples:")
for word in ["diabetes","heart","infection","antibiotics","kidney","stroke"]:
    if word in wv:
        similar = wv.most_similar(word, topn=4)
        print(f" {word:15s}: {[f'{w}({s:.2f})' for w,s in similar]}")

Vocabulary: 222 tokens | Dim: 100

Semantic similarity examples:
 diabetes       : ['education(0.90)', 'glucagon(0.90)', 'dextrose(0.90)', 'hypoglycemia(0.90)']
 heart          : ['diuresis(1.00)', 'bnp(1.00)', 'ejection(1.00)', 'weights(1.00)']
 infection      : ['erythema(0.89)', 'drainage(0.89)', 'warmth(0.89)', 'e(0.88)']
 antibiotics    : ['cellulitis(0.90)', 'drainage(0.89)', 'wound(0.89)', 'warmth(0.89)']
 kidney         : ['damage(0.89)', 'amlodipine(0.88)', 'function(0.88)', 'acute(0.88)']
 stroke         : ['mri(0.89)', 'ischemic(0.89)', 'tpa(0.89)', 'fibrillation(0.88)']


In [21]:
cbow_model = Word2Vec(sentences=augmented, vector_size=100, window=5, min_count=2, workers=4, sg=0, negative=10, epochs=40, seed=42)

wv_cbow = cbow_model.wv
print(f"CBOWVocabulary: {len(wv_cbow.key_to_index)} tokens | Dim: {wv_cbow.vector_size}")
print("\nSemantic similarity examples:")
for word in ["diabetes","heart","infection","antibiotics","kidney","stroke"]:
    if word in wv_cbow:
        similar = wv_cbow.most_similar(word, topn=4)
        print(f" {word:15s}: {[f'{w}({s:.2f})' for w,s in similar]}")

CBOWVocabulary: 222 tokens | Dim: 100

Semantic similarity examples:
 diabetes       : ['dextrose(0.92)', 'glucose(0.92)', 'education(0.92)', 'hypoglycemia(0.92)']
 heart          : ['weights(1.00)', 'bnp(1.00)', 'diuresis(1.00)', 'reduced(1.00)']
 infection      : ['warmth(0.90)', 'drainage(0.89)', 'wound(0.89)', 'post-operative(0.89)']
 antibiotics    : ['wound(0.89)', 'drainage(0.89)', 'cellulitis(0.89)', 'erythema(0.89)']
 kidney         : ['organ(0.92)', 'function(0.92)', 'damage(0.92)', 'amlodipine(0.92)']
 stroke         : ['fibrillation(0.90)', 'atrial(0.90)', 'rate(0.90)', 'prevention(0.89)']
